Abbiamo discusso come BIM sia limitato dal fatto che non prende in considerazione la term frequency e la lunghezza del documento.

Ricordiamo la formula generale che avevamo ricavato con BIM:
$$
RSV_d =
\log \prod_{t_i:\,x_i=y_i=1}
\frac{
\Pr(x_i = 1 \mid R, v_q)\,\Pr(x_i = 0 \mid \bar R, v_q)
}{
\Pr(x_i = 1 \mid \bar R, v_q)\,\Pr(x_i = 0 \mid R, v_q)
}
$$
Usando l'assunzione per cui i documenti rilevanti sono una piccola frazione della collezione, possiamo approssimare il comportamento dei documenti non rilevanti con quello dell’intera collezione:

$$
\Pr(x_i = 1 \mid \bar R, v_q) \approx \Pr(x_i=1)
$$

$$
\Pr(x_i = 0 \mid \bar R, v_q) \approx \Pr(x_i=0)
$$

ottenendo:

$$
RSV_d =
\sum_{t_i:\,x_i=y_i=1}
\log
\frac{
\Pr(x_i = 1 \mid R, v_q)\,\Pr(x_i=0)
}{
\Pr(x_i = 1)\,\Pr(x_i = 0 \mid R, v_q)
}
$$

### Passaggio da BIM a modello con TF
Nel BIM classico per ogni termine i si ha solo $x_i \in \{0,1\}$, indicando se il termine è presente o meno nel documento. Tuttavia, questo approccio non tiene conto di quanto spesso un termine appare in un documento (term frequency). 

Per introdurre la **term frequency TF**, si sostituisce quindi la variabile binaria $x_i$ con una variabile aleatoria di conteggio $d_i =$ #occorrenze del termine $i$ in $d$. Non mi interessa quindi più solo se il termine è presente o meno, ma voglio modellare quante volte compare.

Quindi la formula cambia: 
$$ RSV_d = \sum_{t_i:\,x_i=y_i=1} \log \frac{\Pr(d_i = n_i \mid R, v_q) \Pr(d_i=0)}{\Pr(d_i = n_i) \Pr(d_i = 0 \mid \bar R, v_q)} $$

Per utilizzare la formula quindi dobbiamo scegliere un modello probabilistico per i conteggi, cioè dobbiamo modellare la variabile aleatoria $d_i$. 

Poiché è una v.a. discreta e si vogliono modellare conteggi del tipo $Pr(d_i = k)$ (probabilità che il termine $i$ appaia $k$ volte in un documento), una prima idea è utilizzare una **Binomiale**.

In questo senso si pensa a un documento come un insieme di $l$ posizioni (parole), e **in ogni posizione il termine compare con una certa probabilità $\tilde{p}$. In tal caso $d_i \sim \mathrm{Binomial}(l, \tilde{p})$** e quindi:
$$Pr(d_i = k) = \binom{l}{k} \tilde{p}^k (1-\tilde{p})^{l-k}$$

Tuttavia la distribuzione binomiale ha alcuni problemi: **dipende da l (lunghezza del documento)**, ha formule brutte ed in generale è difficile da usare nel ranking. Per questo motivo si preferisce fare affidamento a un trick matematico e passare alla **Poisson**.

Quando $l$ è grande e $\tilde{p}$ è piccolo, la distribuzione binomiale può essere approssimata con una distribuzione di Poisson di parametro $\lambda = l\tilde{p}$:

$$
\mathrm{Binomial}(l,\tilde{p}) \approx \mathrm{Poisson}(\lambda)
$$

con:

$$
\lambda = \text{valore atteso di occorrenze del termine} = l\tilde{p}
$$
Poisson è migliore della binomiale poiché ha una formula più semplice da maneggiare. Inoltre intuitivamente $\lambda$ è grande se il termine è frequente e piccolo se è raro, quindi è un buon parametro per modellare la frequenza dei termini.

Quindi per stimare la $Pr(d_i = x)$ (il termine $i$ appare $x$ volte in un documento) si avrà:
$$
\Pr(d_i = x) = \frac{\lambda^x e^{-\lambda}}{x!}
$$

#### Stima di $\lambda$ 
Ma come stimare $\lambda$? Una prima idea ragionevole è $$\lambda_j \approx \frac{cf_j}{N}$$ cioè il numero atteso di occorrenze del termine $j$ in un documento è dato approssimativamente dal numero totale di occorrenze del termine $j$ nella collezione ($cf_j$) diviso il numero totale di documenti ($N$).

Tuttavia ci rendiamo conto che **non basta una sola $\lambda$, poiché un termine si comporta diversamente in base al fatto che appartenga a documenti rilevanti o alla collezione in generale** 

(Ossia la frequenza attesa di un termine può cambiare a seconda del tipo di documento che sto guardando: ad es. data la query "health plan", il termine $t_j = \text{health}$ potrebbe apparire in tutta la collezione mediamente poco (es. $\gamma_j = 0.2$), mentre nei documenti rilevanti per la query molto più spesso (es. $\rho_j = 3$ ossia 3 volte per documento)).

Per questo motivo si introduce un modello Poisson con due parametri:
- $\rho_j$ = frequenza attesa del termine $t_j$ nei documenti rilevanti;
- $\gamma_j$ = frequenza attesa del termine $t_j$ nell'intera collezione.

Usando questo modello otteniamo che:
$$\Pr(d_j = n_j \mid R, v_q) = \frac{\rho_j^{n_j} e^{-\rho_j}}{n_j!}$$
$$\Pr(d_j = 0 \mid R, v_q) = e^{-\rho_j}$$
$$\Pr(d_j = n_j) = \frac{\gamma_j^{n_j} e^{-\gamma_j}}{n_j!}$$
$$\Pr(d_j = 0 \mid \bar R, v_q) = e^{-\gamma_j}$$

Sostituendo nella formula del RSV otteniamo:
$$
RSV_d =
\sum_{t_j:\,y_j=1}
\log
\frac{
\frac{\rho_j^{n_j} e^{-\rho_j}}{n_j!} e^{-\gamma_j}
}{
\frac{\gamma_j^{n_j} e^{-\gamma_j}}{n_j!} e^{-\rho_j}
}
=
\sum_{t_j:\,y_j=1}
\log
\frac{\rho_j^{n_j}}{\gamma_j^{n_j}}
=
\sum_{t_j:\,y_j=1}
n_j \log \frac{\rho_j}{\gamma_j}
$$

Analizziamo la formula ottenuta: abbiamo una sommatoria per ogni termine in comune tra query e documento di:
- $n_j$ = quante volte il termine $j$ appare -> ogni occorrenza del termine aumenta il punteggio
- Il peso $log \frac{\rho_j}{\gamma_j}$: rappresenta il rapporto in log tra la frequenza attesa del termine nei rilevanti e la frequenza attesa del termine in generale. Se $\rho_j$ è molto più grande di $\gamma_j$ (il termine è molto più frequente nei rilevanti che in generale) allora il peso è alto, altrimenti se appare più o meno con la stessa frequenza nei rilevanti e in generale allora il rapporto è vicino a 1 e quindi log vicino a 0.

Abbiamo quindi ottenuto una formula del tipo $RSV_d = \sum_{t_j:\,x_j=y_j=1} n_j w_j$, cioè **term frequency (n_j) per un peso w_j che ricorda concettualmente idf (infatti quel rapporto misura quanto un termine è più "speciale" nei documenti rilevanti rispetto al suo comportamento normale nella collezione, con idf similmente il termine pesa tanto se compare in pochi documenti nella collezione)**. Stiamo quindi in un certo senso ricostruendo tf-idf in modo probabilistico, partendo da un modello di base (BIM) e introducendo la term frequency tramite una modellazione Poisson delle occorrenze.

**PROBLEMA**: $\gamma_j$ come visto può essere stimato come $\gamma_j \approx \frac{cf_j}{N}$, ma **per stimare $\rho_j$ (frequenza attesa del termine nei rilevanti) serve sapere quali documenti sono rilevanti per la query, ma questo è esattamente quello che stiamo cercando a priori.** Quindi questo modello non è direttamente utilizzabile.

#### 2-Poisson
Non potendo stimare $\rho_j$ in modo diretto, non utilizzeremo la formula esatta ma manterremo l'intuizione. In particolare dobbiamo ricordare le seguenti tre proprietà fondamentali:
1. **La term frequency di un termine presente nella query e nel documento deve aumentare lo score del documento** (all'aumentare di $n_j$ deve aumentare $RSV_d$)
2. **I termini rari devono pesare di più**
3. **Saturazione**: l'effetto della tf non deve essere lineare e unbounded: da 1 a 2 occorrenze il punteggio deve aumentare molto, da 10 a 11 occorrenze invece non deve aumentare quasi per niente 

Ci accorgiamo anzitutto che $\rho_j$ non è una modellazione propriamente corretta, **il modello di Poisson (da solo) non è propriamente adatto per modellare la term frequency per un termine in un documento rilevante $Pr(d_j = n_j \mid R, v_q)$, infatti**:
- **Contentless terms** (es. stopwords): sono parole che appaiono in quasi tutti i documenti (es. "the", "and", "of") e non descrivono il topic. Poisson è più adatta a questi termini perché le loro occorrenze sono distribuite in modo relativamente stabile e poco dipendente dal topic del documento.
- **Contentful terms**: si tratta dei termini che invece tendono a descrivere i topic del documento (es. cancer, blockchain, health). Questo tipo di parole descrive topic specifici e quindi se appaiono nella query tenderanno ad apparire molto più spesso nei documenti rilevanti che in generale -> avranno una frequenza media più alta nei documenti, dovremmo avere un parametro Poisson più alto

Il punto è che quindi per i contentful terms alcuni documenti avranno 0 occorrenze di essi, altri ne avranno moltissime -> una sola Poisson non è in grado di modellare correttamente questa distribuzione. Avrebbe senso modellare $Pr(d_j = n_j \mid R, v_q)$ non con una singola Poisson, ma con una combinazione di due Poisson distinte...

**Da qui deduciamo che in generale un termine può comportarsi in due modi diversi nei documenti e introduciamo i due stati seguenti di un termine per ogni documento**:
- **Non Elite** se il termine è non rilevante per il documento, compare poco o per niente
- **Elite** se il termine descrive il documento e compare molto spesso.

Per modellare questa situazione, invece di utilizzare una singola distribuzione di Poisson per stimare la term frequency, **definiamo  due distribuzioni di Poisson distinte per i due stati (elite e non elite, da qui 2-Poisson)**: $\mathrm{Poisson}(\mu_j)$ per il termine j elite e $\mathrm{Poisson}(\bar{\mu_j})$ per il termine j non elite. Chiaramente ci aspettiamo, come accennato prima, che $\mu_j$ sia più grande di $\bar{\mu_j}$ (perché i termini elite compaiono più frequentemente nei documenti rilevanti).

Possiamo inoltre modellare l'eliteness per ogni coppia termine-documento come una variabile aleatoria binaria $E_j \in \{0,1\}$ che indica se il termine $j$ è elite o non elite per quel documento.

Applicando queste idee su $Pr(d_j = n_j \mid R, v_q)$ sfruttando la legge della probabilità totale otteniamo:
$$ \Pr(d_j = n_j \mid R, v_q) = \Pr(d_j = n_j \mid E_j) \Pr(E_j \mid R, v_q) + \Pr(d_j = n_j \mid \bar E_j) \Pr(\bar E_j \mid R, v_q) $$
Ossia la probabilità che il termine $j$ appaia $n_j$ volte in un documento rilevante è data (probabilità totale) dalla possibilità che ciò avvenga o nel caso in cui il termine sia elite o nel caso in cui il termine sia non elite. 

Per pulire le formule denotiamo con $p_j = \Pr(E_j \mid R, v_q)$ la probabilità che il termine $j$ sia elite in un documento rilevante, usando Poisson quindi:
$$ \Pr(d_j = n_j \mid R, v_q) = \mathrm{Poisson}(n_j; \mu_j) p_j + \mathrm{Poisson}(n_j; \bar{\mu_j}) (1-p_j) = p_j \frac{e^{-\mu_j} \mu_j^{n_j}}{n_j!} + (1-p_j) \frac{e^{-\bar{\mu_j}} \bar{\mu_j}^{n_j}}{n_j!} $$
dove ricordiamo $\mu_j$ è la frequenza attesa del termine $j$ nei documenti rilevanti se è elite, mentre $\bar{\mu_j}$ è la frequenza attesa del termine $j$ nei documenti rilevanti se è non elite.

Quindi in generale stiamo pensando ad ogni documento come uan combinazione di termini: una parte non-elite (rumore) e una parte elite (significativa per il documento). 

Il seguente grafico mostra ancora di più la necessità di modellare $Pr(d_j = n_j \mid R, v_q)$ con una combinazione di due Poisson distinte (2-Poisson). Sull'asse x numero di occorrenze del termine, sull'asse y la probabilità che occorra. Il grafico è su un singolo termine, la curva blu mostra la distribuzione di Poisson per i documenti per cui il termine non è elite. In questo caso osserviamo come la probabilità che il termine compaia poche volte è molto più alta di quella che compaia molte volte. La curva arancione invece mostra la distribuzione di Poisson per i documenti per cui il termine è elite, osserviamo in questo caso il comportamento opposto (è più probabile che il termine compaia molte volte). La curva verde è la combinazione delle due, e tiene quindi conto sia dei documenti per cui il termine è elite che di quelli per cui non è elite.

<img src="img/2pois.png" width="400">

Definiamo sempre per pulire le formule mannaggia alla madonna:
$$C(n_j) = \mathrm{Poisson}(n_j; \mu_j) = \text{probabilità che il termine $j$ appaia $n_j$ volte in un documento rilevante se il termine è elite per il documento}$$
$$\bar{C}(n_j) = \mathrm{Poisson}(n_j; \bar{\mu_j}) = \text{probabilità che il termine $j$ appaia $n_j$ volte in un documento rilevante se il termine non è elite per il documento}$$
$$p_j = \Pr(E_j \mid R, v_q) = \text{probabilità che il termine $j$ sia elite in un documento rilevante}$$
$$\bar{p_j} = \Pr(E_j) = \text{probabilità che il termine $j$ sia elite in un documento nell'intera collezione}$$

Allora possiamo riscrivere la formula del RSV come segue (per un singolo termine della query presente anche nel documento, sarebbe da sommare su tutti i termini in comune tra query e documento):

<img src="img/PORCOIDDIOOO.png" width="400">

Per stimare gli score quindi sarebbe necessario stimare anche i parametri $\mu_j$, $\bar{\mu_j}$, $p_j$ e $\bar{p_j}$, ma anche in questo caso è difficile farlo in pratica.

**Mantenendo queste stesse intuizioni, da qui nasce l'idea verso BM25**.

#### Verso BM25
2-Poisson è troppo complicato per essere implementato -> teniamo solo il comportamento che ci interessa, in particolare come accennato prima:
1. Se $n_j$ = 0$ (il termine non compare nel documento) allora il termine non contribuisce allo score del documento
2. Al crescere di $n_j$ (il termine compare più volte) lo score del documento aumenta, **ma non deve crescere all'infinito, deve saturare.**

Perché deve saturare? **Intuitivamente perché TF è un'evidenza di eliteness di un termine per un documento -> se $n_j$ è 0 allora nessuna evidenza, se $n_j$ maggiore evidenza, ma se $n_j$ è molto grande allora è già chiaro da prima che il termine è elite**.

Serve quindi una funzione che cresca inizialmente ma sia bounded, appiattendosi all'aumentare di $n_j$, sia $k \gt 0$ costante, allora:
$$\frac{(k+1) n_j}{k + n_j}$$
Questa funzione è perfetta in questo senso perché **fintanto che $n_j$ è piccolo allora la funzione cresce quasi linearmente, ma per n_j che tende a $\infty$ la funzione si avvicina a $k+1$ e si ferma** (perché quando $n_j$ diventa enorme allora il termine k al denominatore conta sempre meno rispetto a n_j, quindi intuitivamente $k+n_j \approx n_j$ e allora la formula diventa approssimativamente $k+1$). La costante $k$ rappresenta quindi una costante di saturazione che regola quanto velocemente la funzione si appiattisce. 

Esempio con $k=2$:

$$
f(n_j)=\frac{(2+1)n_j}{2+n_j}=\frac{3n_j}{2+n_j}
$$

Se $n_j=1$:

$$
f(1)=\frac{3}{3}=1
$$

Se $n_j=2$:

$$
f(2)=\frac{6}{4}=1.5
$$

Se $n_j=10$:

$$
f(10)=\frac{30}{12}=2.5
$$

Se $n_j=100$:

$$
f(100)=\frac{300}{102}\approx 2.94
$$

Il limite è:

$$
\lim_{n_j\to\infty}\frac{3n_j}{2+n_j}=3=k+1
$$

Quindi la funzione cresce, ma sempre più lentamente: da 1 a 2 occorrenze il contributo aumenta molto, mentre da 100 a 101 aumenta quasi niente. Questo rappresenta la saturazione della term frequency.

Abbiamo quindi definito una funzione ragionevole per misurare **quanto è forte l'evidenza che un termine $j$ sia elite per il documento**. Ci manca ora da definire **quanto è probabile che effettivamente $j$ fosse elite per quel documento**. 

Per fare ciò compariamo le probabilità che il termine $j$ sia elite in un documento rilevante ($p_j$) e che sia elite in un documento generico nella collezione ($\bar{p_j}$). Se $p_j$ è molto più grande di $\bar{p_j}$ allora il fatto che il termine $j$ sia elite per quel documento è un forte indicatore di rilevanza (perché la probabilità che fosse elite in un documento generico è molto più bassa). 

Confrontando le due odds $\frac{p_j}{1-p_j}$ e $\frac{\bar{p_j}}{1-\bar{p_j}}$ si arriva a $$\log \frac{p_j (1-\bar{p_j})}{\bar{p_j} (1-p_j)}$$
Si torna poi al logaritmo per gli stessi motivi visti in precedenza (trasforma i prodotti in somme e rende il peso interpretabile, se il rapporto è 1 il log è 0 mentre se è maggiore di 1 il log è positivo e se è minore di 1 il log è negativo): $$w_j = \log \frac{p_j (1-\bar{p_j})}{\bar{p_j} (1-p_j)}$$ Questo fattore è detto **Binary Probabilistic Weight**.

Quindi la 2-Poisson è sostituita dalla formula seguente (ricorda sempre che è da mettere nella sommatoria su tutti i termini in comune tra query e documento):
$$\frac{(k+1) n_j}{k + n_j} \log \frac{p_j (1-\bar{p_j})}{\bar{p_j} (1-p_j)}$$

Anche qui torna la somiglianza con tf-idf: il primo termine modella una tf saturata che ci dice quanto il documento parla di quel dato termine mentre il secondo è un peso probabilistico simil idf, che ci dice quanto quel termine è informativo.

Il problema qui di nuovo sta in $p_j$ e $\bar{p_j}$, che non sono stimabili a priori nell'ad-hoc retrieval -> approssimazioni.

Riassumendo il percorso logico:

1. **BIM**: il modello parte da una rappresentazione binaria dei termini. Per ogni termine si considera solo se esso è presente o assente nel documento. Quindi il contributo di un termine allo score è di tipo 0/1: se il termine della query compare nel documento contribuisce, altrimenti no.

2. **Poisson**: per superare il limite binario del BIM, si introduce una variabile di conteggio \(n_j\), cioè il numero di occorrenze del termine \(t_j\) nel documento. In questo modo il modello inizia a considerare la **term frequency**: un termine che compare più volte fornisce più evidenza che il documento tratti quell’argomento.

3. **2-Poisson**: una singola Poisson non descrive bene il comportamento dei termini più informativi, perché un termine può essere poco presente nei documenti in cui non è centrale e molto presente nei documenti in cui descrive davvero il topic. Si introduce quindi l’idea di **eliteness**: un termine può essere elite o non-elite per un documento. Da qui nasce l’intuizione che la term frequency debba aumentare lo score, ma con **rendimenti decrescenti**, cioè con saturazione.

4. **Formula approssimata**: poiché il modello 2-Poisson completo è troppo complesso da stimare direttamente, se ne mantiene solo il comportamento desiderato. Il contributo di un termine viene quindi modellato come prodotto tra:
   - una componente di **term frequency saturata**, che cresce con \(n_j\) ma tende a un limite;
   - un **peso probabilistico/idf-like**, che misura quanto il termine è informativo per distinguere documenti rilevanti da documenti generici.

In sintesi, il percorso porta da un modello binario a una funzione di ranking più realistica: $$\text{TF Saturata} \times \text{Peso Probabilistico}$$
 
Questa è l’idea che verrà mantenuta e resa pratica da BM25.




#### BM25
**Ad-hoc retrieval** == situazione "normale" di ricerca, in cui arriva una query nuova e il sistema deve recuperare documenti rilevanti senza avere già i giudizi di rilevanza per quella query.

Ora quindi come anticipato il problema è che in ad-hoc retrieval non sappiamo quali documenti siano rilevanti per una query, né tantomeno sappiamo quali termini siano elite per certi documenti. Per tali ragioni, $p_j$ (probabilità che il termine $j$ sia elite in un documento rilevante) e $\bar{p_j}$ (probabilità che il termine $j$ sia elite in un documento generico nella collezione) non sono stimabili a priori -> il peso probabilistico $log \frac{p_j (1-\bar{p_j})}{\bar{p_j} (1-p_j)}$ non è stimabile direttamente.

Per ottenere quindi una formula utilizzabile ricorriamo nuovamente a delle **approssimazioni**:
1. **Non sapendo quanto sia probabile che un termine sia elite nei documenti rilevanti, si assume brutalmente un valore neutro** $p_j = Pr(E_j \mid R, v_q) \approx 0.5$. 
2. **Si assume ancora più brutalmente che la probabilità che un termine sia elite in un documento generico della collezione sia approssimativamente data dalla frequenza del termine nella collezione** $\bar{p_j} = Pr(E_j) \approx \frac{df_j}{N}$, dove $df_j$ è il numero di documenti in cui compare il termine $j$ e $N$ è il numero totale di documenti nella collezione. L'idea dietro questa approssimazione è che si identifica l'eliteness con la semplice presenza del termine: secondo questa logica quindi se un termine compare in molti documenti della collezione allora è più probabile che sia elite in un documento generico.

Ora lavoriamo sulla formula del peso probabilistico:

$$
\log \frac{p_j (1-\bar{p}_j)}{\bar{p}_j (1-p_j)}
=
\log \frac{p_j}{1-p_j}
+
\log \frac{1-\bar{p}_j}{\bar{p}_j}
$$

Se $p_j = 0.5$ allora $\log \frac{p_j}{(1-p_j)} = \log \frac{0.5}{0.5} = \log 1 = 0$, quindi resta solo $\log \frac{1-\bar{p_j}}{\bar{p_j}}$. 

Sostituendo quindi $\bar{p_j} = \frac{df_j}{N}$ otteniamo:
$$\log \frac{1-\bar{p_j}}{\bar{p_j}} = \log \frac{1-\frac{df_j}{N}}{\frac{df_j}{N}} = \log \frac{N-df_j}{df_j}$$

**Assumendo che la document frequency di un termine $df_j$ sia molto più piccola del numero totale di documenti $N$** (assunzione ragionevole per i termini più informativi), allora $N-df_j \approx N$, e quindi:
$$\log \frac{N-df_j}{df_j} \approx \log \frac{N}{df_j}$$
**e questa è proprio la formula dell'idf!**

Combinando i due pezzi si ottiene quindi:
$$RSV_d = \sum_{t_j:\,y_j=1} \frac{(k+1) n_j}{k + n_j} \log \frac{N}{df_j}$$
Per cui comunque valgono i due concetti chiave:
- la tf è importante per lo score, ma deve saturare
- i termini rari (con basso df) devono pesare di più perché più informativi


La prima forma concreta di BM25 è proprio quella appena vista:
$$RSV_d = \sum_{t \in q} \frac{(k+1) tf_{t,d}}{k + tf_{t,d}} \log \frac{N}{df_t}$$
dove il primo termine è la tf saturata e il secondo è l'idf.

Come già accennato prima $k$ è **un parametro costante che serve a decidere quanto velocemente la tf deve saturare**. Se ad esempio $k$ è piccolo (es. 0.5) allora la saturazione avverrà molto velocemente, quindi stiamo dicendo che la tf di un termine conta poco (poche occorrenze sono sufficienti per avere un contributo alto). Al contrario $k$ grande ci dice che anche più occorrenze continuano ad aumentare lo score. 

Dal seguente grafico si osserva in ogni curva $\frac{(k+1) n_j}{k + n_j}$ con $k$ diverso, si osserva come il comportamento generale sia una crescita rapida per le prime occorrenze del termine per poi appiattirsi avvicinandosi a $k+1$ per $n_j$ che tende a $\infty$. Le curve più alte sono quindi chiaramente quelle con k maggiore (la più alta blu chiaro è per $k=3$, per $n_j$ che tende a infinito si ha lo score che tende a $3+1=4$)

<img src="img/bm25tf.png" width="400" height="250">

Tipicamente il valore di $k$ è scelto tra 1.2 e 2. Inoltre nota che il fatto di avere al numeratore $k+1$ non cambia il ranking, ma serve solo a normalizzare lo score, facendo si che quando la tf è 1 allora lo score è 1.

**Esercizio**: 
- interpreta la formula per $k=0$ -> in questo caso sostituendo nella formula si ottiene $\frac{(0+1) tf}{0 + tf} = \frac{tf}{tf} = 1$, quindi in questo modo il numero di occorrenze non contribuisce allo score, basta che il termine appaia -> **SI TORNA AL BIM**
- per $k = 1$ -> in questo caso si ha $\frac{(1+1) tf}{1 + tf} = \frac{2 tf}{1 + tf}$, quindi lo score cresce all'aumentare di tf ma satura velocemente, al più si avvicina a 2
- per $k \to \infty$ -> in questo caso si ha $\frac{(k+1) tf}{k + tf} \approx = tf$, quindi in questo caso la tf non satura, ma cresce linearmente all'aumentare di tf

A questo punto, per arrivare a BM25 completo, ci manca la **normalizzazione per la lunghezza del documento**. Infatti come già detto nei documenti più lunghi la tf tende ad essere più alta, ma questo non significa necessariamente che quei documenti siano più rilevanti.

Un documento può essere lungo per due principali motivi: è **verboso** (ripete tanto le stesse cose quindi tf artificialmente alto) oppure perché **ha scope maggiore** (tratta molti argomenti).

Sia $L_d = \sum_{t \in d} tf_{t,d}$ la lunghezza del documento $d$. Sia poi $L_{avg} = \frac{1}{N} \sum_{d \in C} L_d$ la lunghezza media dei documenti nella collezione. 

Definiamo il fattore $$B = (1-b) + b \frac{L_d}{L_{avg}}$$
Tale fattore entra nel denominatore della parte TF di BM25: il parametro $b$ controlla quanto vogliamo penalizzare (normalizzare) i documenti lunghi:
- se $b=0$ allora $B=1$ e quindi non c'è normalizzazione 
- se $b=1$ si ha normalizzazione piena: $B = \frac{L_d}{L_{avg}}$ e quindi avremo al denominatore qualcosa che è grande se il documento è più lungo della media -> score penalizzato per documenti più lunghi (e aumentato per documenti più corti, al denominatore avrei qualcosa di $\lt 1$!). 

**La formula finale di BM25 è quindi la seguente**:
$$RSV_d = \sum_{t \in q} \frac{(k+1) tf_{t,d}}{k B + tf_{t,d}} \log \frac{N}{df_t} = \sum_{t \in q} \frac{(k+1) tf_{t,d}}{k ((1-b) + b \frac{L_d}{L_{avg}}) + tf_{t,d}} \log \frac{N}{df_t}$$

Il parametro $b$, così come $k$, è un parametro empirico da ottimizzare via benchmark in base alla collezione e al tipo di query. Tipicamente comunque $k$ è scelto tra 1.2 e 2, mentre $b \approx 0.75$ è un buon compromesso.

Come anticipato se la lunghezza del documento è maggiore della media $L_d \gt L_{avg}$ allora il denominatore aumenta e quindi il contributo del termine diminuisce, mentre se la lunghezza del documento è minore della media $L_d \lt L_{avg}$ allora il denominatore diminuisce e quindi il contributo del termine 
aumenta.

Si osserva dalla figura proprio questo fenomeno: al crescere di $b$ sull'asse x (**parametro di normalizzazione, più è alto più lo score è sensibile alla lunghezza del documento**) lo score aumenta proprio per i documenti più corti della media.

<img src="img/bm25_len.png" width="400" height="200">

**Esercizio**:
- interpretare la formula BM25 per $k=0$ -> in questo caso si ha $\frac{(0+1) tf}{0 B + tf} = \frac{tf}{tf} = 1$, quindi si torna al BIM (comportamento binario, conta solo se il termine è presente o meno, non conta la tf)
- interpretare la formula BM25 per $b=0$ -> in questo caso si ha $B = (1-0) + 0 \frac{L_d}{L_{avg}} = 1$, quindi non c'è normalizzazione per la lunghezza del documento, si torna al caso $\frac{(k+1) tf}{k + tf}$
- $k=1$ e $b=0$ -> con $b=0$ non c'è normalizzazione per lunghezza, si torna al caso $\frac{2 tf}{1 + tf}$ quindi si ha un tf che conta ma satura velocemente
- $k \to \infty$ e $b=0$ -> in questo caso di nuovo niente normalizzazione, si ha $\frac{(k+1) tf}{k + tf} \approx tf$, quindi si ha un tf che non satura ma cresce linearmente
- $k \to \infty$ e $b=1$ -> in questo caso si ha normalizzazione piena per la lunghezza e tf -> $\frac{(k+1) tf}{k \frac{L_d}{L_{avg}} + tf} \approx tf \frac{L_{avg}}{L_d}$, quindi si ha un tf che non satura ma cresce linearmente e normalizzazione piena per la lunghezza del documento

#### Conclusione su BM25
Si osserva anzitutto un confronto tra BM25 e tf-idf classico:

Si osserva come con tf-idf, anche usando $1 + \log(1+tf)$, il termine learning ripetuto 1024 volte da ancora tanto peso al documento 1, mettendolo in ranking sopra a doc 2 nonostante contenesse solo una volta il termine machine. Al contrario BM25 bilancia molto meglio questo aspetto: la saturazione per il documento 1 avviene molto più rapidamente usando $k=2$ e permette un ranking più adeguato.

<img src="img/bm_25_tf_idf.png" width="400" height="200">

Si osserva inoltre che **ha senso aggiungere allo scorer un ulteriore peso per la frequenza di termini nella query**, con una nuova costante $k_2$ che regola la saturazione di un termine nella query: $\frac{(k_2)tf_{t,q}}{k_2 + tf_{t,q}}$.

Questo fattore aggiuntivo non cambia molto per query brevi, in quanto in queste tipicamente i termini compaiono una sola volta. **Piuttosto è utile per query lunghe, in cui se un termine ricorre più spesso nella query evidentemente significa che è più importante degli altri**.

Chiaramente inoltre la normalizzazione per la lunghezza della query non avrebbe senso perché **la query è fissa quando si stanno ordinando i documenti** (tutti i documenti sono confrontati rispetto alla stessa query)

$$RSV_d = \sum_{t \in q} (\log \frac{N}{df_t}) (\frac{(k_1+1) tf_{t,d}}{k_1 ((1-b) + b \frac{L_d}{L_{avg}}) + tf_{t,d}}) (\frac{(k_2+1)tf_{t,q}}{k_2 + tf_{t,q}})$$

**Ma in generale, quale ranking model usare quindi?**
- se si vuole un modello semplice -> Vector Space Model + tf-idf
- se si vuole un primo ranking robusto -> BM25 con parametri correttamente ottimizzati (tuned)
- nel mezzo: BM25 o Language Models con parametri non ottimizzati (impostati a valori di default) oppure solo alcuni ottimizzati

Oggi BM25 ancora un modello molto utilizzato in diverse applicazioni per l'ad hoc retrieval, soprattutto in contesti dove è importante avere efficienza.

Sicuramente BM25 infatti è più debole rispetto a un modello a embedding: mentre BM25 se cerco "cane" non capisce che un documento con solo "cavalier king" potrebbe essere rilevante (si basa solo sulla presenza lessicale dei termini) il modello di embedding dovrebbe avere vicinanza semantica tra i due e capire il collegamento. **Però BM25 ha il vantaggio enorme di essere molto più efficiente in quanto funziona tramite inverted index!** Quando arriva una query, BM25 non la confronta con tutti i documenti della collezione, ma solo con quelli che contengono almeno uno di quei termini. Un modello a embedding al contrario rappresenta query e documenti con vettori densi -> per trovare i documenti simili deve fare moltissimi confronti e anche con i dovuti algoritmi di ottimizzazione tipo ANN (Approximate Nearest Neighbors) o HNSW comunque impiega molto più tempo e risorse in memoria rispetto a BM25. 



**Domanda esame: dati 1000 documenti, data la formula BM25, come costruisco il motore di retrieval?**
1. Per prima cosa devo costruire l'inverted index. Quindi preprocessing della collezione, per ogni documento tokenizzazione ed eventuale rimozione stopwords e stemming. 
2. uso algoritmi (es. BSBI o SPIMI) e ottengo l'inverted index.
3. Per calcolare BM25 è necessario avere a disposizione statistiche, in particolare $N$ (numero totale di documenti, in questo caso 1000), $df_t$ (document frequency **per ogni termine**, equivale alla lunghezza della posting list del termine), $L_d$ (lunghezza di ogni documento) e $L_{avg}$ (lunghezza media dei documenti). 

Quando arriva una query si procede come segue. Immaginiamo q = "machine learning". Allora anzitutto si **recuperano i candidati facendo OR delle posting list dei termini della query** (in questo caso unione posting list di "machine" e "learning"). Per ogni documento candidato (posting) si calcola:
$$RSV_d = \sum_{t \in q} (\log \frac{N}{df_t}) (\frac{(k_1+1) tf_{t,d}}{k_1 ((1-b) + b \frac{L_d}{L_{avg}}) + tf_{t,d}}) (\frac{(k_2+1)tf_{t,q}}{k_2 + tf_{t,q}})$$
(quindi per ogni termine della query che compare anche nel documento si calcola il contributo di quel termine e si sommano tutti. Si noti che se il termine della query non compare nel documento candidato allora il suo contributo nella sommatoria è zero).

Una volta ottenuti gli $RSV_d$ per tutti i documenti candidati, si ordinano in ordine decrescente e si restituiscono i top-k risultati.
